In [1]:
# Cell 1 - Imports
# Core data stack
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import joblib

# ML libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, IsolationForest

# Imbalance + explainability + deployment
from imblearn.under_sampling import RandomUnderSampler
import shap
import onnx
import onnxruntime as ort
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

warnings.filterwarnings("ignore")
np.random.seed(42)

print("[Cell 1/13] Imports complete.")

c:\Users\Lenovo\Codes\NetworkAnomolyDetection\ids_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Cell 1/13] Imports complete.


In [2]:
# Cell 2 - Load Data
try:
    print("[Cell 2/13] Loading cicids_clean.parquet...")
    data_path = Path("cicids_clean.parquet")
    if not data_path.exists():
        raise FileNotFoundError(f"Dataset not found at: {data_path.resolve()}")

    df = pd.read_parquet(data_path)
    print(f"Loaded dataset shape: {df.shape}")

    required_targets = {"label", "classlabel"}
    missing_targets = required_targets - set(df.columns)
    if missing_targets:
        raise ValueError(f"Missing required target columns: {missing_targets}")

    y = df["label"].astype(str)

    # Drop labels from features, coerce numeric, clean inf/nan, and force float32 for memory efficiency.
    X = df.drop(columns=["label", "classlabel"], errors="ignore")
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(np.float32)

    print(f"Feature matrix shape: {X.shape}")
    print(f"Target vector shape: {y.shape}")
    print(f"Feature dtype summary: {X.dtypes.unique()}")
except Exception as e:
    print(f"[ERROR][Cell 2] {e}")
    raise

[Cell 2/13] Loading cicids_clean.parquet...
Loaded dataset shape: (9167271, 59)
Feature matrix shape: (9167271, 57)
Target vector shape: (9167271,)
Feature dtype summary: [dtype('float32')]


In [3]:
# Cell 3 - Handle Class Imbalance (targeted undersampling)
try:
    print("[Cell 3/13] Applying class-wise undersampling...")

    class_counts = y.value_counts()
    print("Original class distribution (top 10):")
    print(class_counts.head(10))

    sampled_parts = []
    for cls, count in class_counts.items():
        cls_df = X.loc[y == cls].copy()
        cls_df["label"] = cls

        if cls == "Benign":
            n_keep = min(150_000, count)
            cls_df = cls_df.sample(n=n_keep, random_state=42)
        elif count > 50_000:
            cls_df = cls_df.sample(n=50_000, random_state=42)

        sampled_parts.append(cls_df)

    balanced_df = pd.concat(sampled_parts, axis=0, ignore_index=True)
    balanced_df = balanced_df.sample(frac=1.0, random_state=42).reset_index(drop=True)

    y_balanced = balanced_df["label"].astype(str)
    X_balanced = balanced_df.drop(columns=["label"]).astype(np.float32)

    print(f"Balanced dataset shape: {X_balanced.shape}")
    print("Balanced class distribution (top 10):")
    print(y_balanced.value_counts().head(10))
except Exception as e:
    print(f"[ERROR][Cell 3] {e}")
    raise

[Cell 3/13] Applying class-wise undersampling...
Original class distribution (top 10):
label
Benign            7185881
DDoS-LOIC-HTTP     575364
DoS-Hulk           318740
DDoS-HOIC          198861
Botnet             145968
DDoS               128062
DDoS-NTP           121326
DDoS-TFTP           98833
Bruteforce-SSH      97260
Infiltration        94857
Name: count, dtype: int64
Balanced dataset shape: (799795, 57)
Balanced class distribution (top 10):
label
Benign            150000
DDoS-HOIC          50000
DoS-Hulk           50000
DDoS               50000
DDoS-NTP           50000
Botnet             50000
Bruteforce-SSH     50000
DDoS-LOIC-HTTP     50000
Infiltration       50000
DDoS-TFTP          50000
Name: count, dtype: int64


In [4]:
# Cell 4 - Feature Selection
try:
    print("[Cell 4/13] Training quick RandomForest for feature importance...")

    rf_fs = RandomForestClassifier(
        n_estimators=20,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced_subsample"
    )
    rf_fs.fit(X_balanced, y_balanced)

    importances = pd.Series(rf_fs.feature_importances_, index=X_balanced.columns)
    selected_features = importances.sort_values(ascending=False).head(20).index.tolist()

    X_selected = X_balanced[selected_features].copy()

    with open("selected_features.txt", "w", encoding="utf-8") as f:
        for feat in selected_features:
            f.write(f"{feat}\n")

    print("Selected top 20 features:")
    print(selected_features)
    print("Saved to selected_features.txt")
except Exception as e:
    print(f"[ERROR][Cell 4] {e}")
    raise

[Cell 4/13] Training quick RandomForest for feature importance...
Selected top 20 features:
['init_fwd_win_bytes', 'avg_fwd_segment_size', 'fwd_seg_size_min', 'fwd_packet_length_mean', 'init_bwd_win_bytes', 'fwd_header_length', 'flow_bytes/s', 'packet_length_max', 'subflow_fwd_bytes', 'avg_packet_size', 'flow_iat_std', 'fwd_packet_length_max', 'fwd_packets/s', 'fwd_packets_length_total', 'packet_length_mean', 'bwd_header_length', 'flow_packets/s', 'flow_iat_max', 'flow_iat_mean', 'fwd_act_data_packets']
Saved to selected_features.txt


In [5]:
# Cell 5 - Train/Test Split
try:
    print("[Cell 5/13] Performing stratified 80/20 split...")

    if "flow_packets/s" not in X_balanced.columns:
        raise KeyError("Required feature 'flow_packets/s' not found in dataset.")

    flow_packets_s = X_balanced["flow_packets/s"].astype(np.float32)

    X_train, X_test, y_train, y_test, flow_packets_s_train, flow_packets_s_test = train_test_split(
        X_selected,
        y_balanced,
        flow_packets_s,
        test_size=0.2,
        random_state=42,
        stratify=y_balanced
    )

    print("Train class distribution (top 10):")
    print(y_train.value_counts().head(10))
    print("\nTest class distribution (top 10):")
    print(y_test.value_counts().head(10))

    label_encoder = LabelEncoder()
    y_train_enc = label_encoder.fit_transform(y_train)
    y_test_enc = label_encoder.transform(y_test)

    joblib.dump(label_encoder, "label_encoder.pkl", compress=3)
    print("Saved label_encoder.pkl")
except Exception as e:
    print(f"[ERROR][Cell 5] {e}")
    raise

[Cell 5/13] Performing stratified 80/20 split...
Train class distribution (top 10):
label
Benign            120000
DDoS-TFTP          40000
DDoS               40000
Bruteforce-SSH     40000
Infiltration       40000
DoS-Hulk           40000
DDoS-LOIC-HTTP     40000
DDoS-HOIC          40000
DoS-Goldeneye      40000
Botnet             40000
Name: count, dtype: int64

Test class distribution (top 10):
label
Benign            30000
DoS-Hulk          10000
DDoS-LOIC-HTTP    10000
DoS-Goldeneye     10000
Infiltration      10000
DDoS-TFTP         10000
DDoS-NTP          10000
Bruteforce-SSH    10000
DDoS-HOIC         10000
DDoS              10000
Name: count, dtype: int64
Saved label_encoder.pkl


In [6]:
# Cell 6 - Scaling
try:
    print("[Cell 6/13] Fitting StandardScaler on training data only...")

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
    X_test_scaled = scaler.transform(X_test).astype(np.float32)

    joblib.dump(scaler, "scaler.pkl", compress=3)
    print("Saved scaler.pkl")
    print(f"Scaled train shape: {X_train_scaled.shape}, dtype: {X_train_scaled.dtype}")
    print(f"Scaled test shape: {X_test_scaled.shape}, dtype: {X_test_scaled.dtype}")
except Exception as e:
    print(f"[ERROR][Cell 6] {e}")
    raise

[Cell 6/13] Fitting StandardScaler on training data only...
Saved scaler.pkl
Scaled train shape: (639836, 20), dtype: float32
Scaled test shape: (159959, 20), dtype: float32


In [16]:
# Cell 7 - Train Random Forest + Metrics
try:
    print("[Cell 7/13] Training RandomForestClassifier...")

    # Edge profile: slightly smaller forest to satisfy Raspberry Pi RAM constraints.
    rf_model = RandomForestClassifier(
        n_estimators=35,
        max_depth=12,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
    rf_model.fit(X_train_scaled, y_train_enc)

    # Save sklearn RF too (useful for SHAP in predictor.py).
    joblib.dump(rf_model, "rf_model.pkl", compress=3)

    y_pred_enc = rf_model.predict(X_test_scaled)
    y_pred = label_encoder.inverse_transform(y_pred_enc)

    print("Classification report:")
    print(classification_report(y_test, y_pred, digits=4, zero_division=0))

    classes = label_encoder.classes_
    cm = confusion_matrix(y_test, y_pred, labels=classes)

    print("False Positive Rate (FPR) per class:")
    total = cm.sum()
    for i, cls in enumerate(classes):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = total - (tp + fp + fn)
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        print(f"{cls}: {fpr:.6f}")

    print("Saved rf_model.pkl")
except Exception as e:
    print(f"[ERROR][Cell 7] {e}")
    raise

[Cell 7/13] Training RandomForestClassifier...
Classification report:
                      precision    recall  f1-score   support

              Benign     0.9790    0.5064    0.6675     30000
              Botnet     1.0000    0.9791    0.9894     10000
      Bruteforce-FTP     0.9992    0.9891    0.9941      1197
      Bruteforce-SSH     1.0000    0.9942    0.9971     10000
                DDoS     0.9967    0.9992    0.9980     10000
            DDoS-DNS     0.7302    0.4387    0.5481       734
        DDoS-Ddossim     0.9642    0.9990    0.9813      1023
           DDoS-HOIC     0.9975    1.0000    0.9988     10000
           DDoS-LDAP     0.6148    0.7368    0.6703       418
      DDoS-LOIC-HTTP     0.7685    0.9963    0.8677     10000
          DDoS-MSSQL     0.9097    0.9665    0.9373      2357
            DDoS-NTP     0.9989    0.9944    0.9966     10000
        DDoS-NetBIOS     0.6000    0.8444    0.7015       135
           DDoS-SNMP     0.6487    0.9256    0.7628       403

In [17]:
# Cell 8 - Train Isolation Forest on Benign Traffic Only
try:
    print("[Cell 8/13] Training IsolationForest on benign training traffic...")

    benign_mask = (y_train == "Benign").values
    X_benign_train = X_train_scaled[benign_mask]

    if X_benign_train.shape[0] == 0:
        raise ValueError("No benign samples found in training set.")

    iso_model = IsolationForest(
        contamination=0.05,
        random_state=42,
        n_estimators=100,
        n_jobs=-1
    )
    iso_model.fit(X_benign_train)

    # Reference range for anomaly score normalization
    iso_reference = -iso_model.decision_function(X_train_scaled)
    iso_min = float(np.min(iso_reference))
    iso_max = float(np.max(iso_reference))

    joblib.dump(iso_model, "iso_model.pkl", compress=3)
    print(f"IsolationForest trained on {X_benign_train.shape[0]} benign rows.")
    print("Saved iso_model.pkl")
except Exception as e:
    print(f"[ERROR][Cell 8] {e}")
    raise

[Cell 8/13] Training IsolationForest on benign training traffic...
IsolationForest trained on 120000 benign rows.
Saved iso_model.pkl


In [18]:
# Cell 9 - Combined Risk Scorer (RF + IF + Statistical Z-score)
try:
    print("[Cell 9/13] Building hybrid risk scorer...")

    benign_index = np.where(label_encoder.classes_ == "Benign")[0]
    if len(benign_index) == 0:
        raise ValueError("'Benign' class not found in label encoder.")
    benign_index = int(benign_index[0])

    flow_mu = float(flow_packets_s_train.mean())
    flow_sigma = float(flow_packets_s_train.std()) + 1e-8

    def compute_risk_scores(X_scaled_batch: np.ndarray, flow_packets_batch: np.ndarray):
        proba = rf_model.predict_proba(X_scaled_batch)
        rf_attack_prob = 1.0 - proba[:, benign_index]

        iso_raw = -iso_model.decision_function(X_scaled_batch)
        iso_norm = (iso_raw - iso_min) / (iso_max - iso_min + 1e-8)
        iso_norm = np.clip(iso_norm, 0.0, 1.0)

        z = np.abs((flow_packets_batch - flow_mu) / flow_sigma)
        z_norm = np.clip(z / 3.0, 0.0, 1.0)

        risk_01 = 0.5 * rf_attack_prob + 0.3 * iso_norm + 0.2 * z_norm
        risk_100 = np.clip(risk_01 * 100.0, 0.0, 100.0)

        status = np.where(
            risk_100 < 40,
            "clean",
            np.where(risk_100 < 70, "suspicious", "threat")
        )
        return risk_100, status, rf_attack_prob, iso_norm, z_norm

    sample_n = min(10, X_test_scaled.shape[0])
    sample_idx = np.arange(sample_n)
    risk_score, risk_status, rf_component, iso_component, z_component = compute_risk_scores(
        X_test_scaled[sample_idx],
        flow_packets_s_test.iloc[sample_idx].values.astype(np.float32)
    )

    risk_df = pd.DataFrame({
        "predicted_label": label_encoder.inverse_transform(rf_model.predict(X_test_scaled[sample_idx])),
        "risk_score": np.round(risk_score, 2),
        "risk_status": risk_status,
        "rf_component": np.round(rf_component, 3),
        "if_component": np.round(iso_component, 3),
        "z_component": np.round(z_component, 3)
    })

    print("Sample risk scores (10 rows):")
    print(risk_df)
except Exception as e:
    print(f"[ERROR][Cell 9] {e}")
    raise

[Cell 9/13] Building hybrid risk scorer...
Sample risk scores (10 rows):
  predicted_label  risk_score risk_status  rf_component  if_component  \
0        DoS-Hulk       65.65  suspicious         0.992         0.499   
1       DDoS-LDAP       74.33      threat         0.970         0.828   
2  DDoS-LOIC-HTTP       53.71  suspicious         0.992         0.100   
3          Benign       23.12       clean         0.386         0.091   
4   DoS-Goldeneye       52.63  suspicious         0.953         0.129   
5       DDoS-LDAP       97.75      threat         1.000         0.925   
6   DoS-Goldeneye       54.23  suspicious         0.975         0.146   
7    Infiltration       42.02  suspicious         0.732         0.144   
8       DDoS-TFTP       64.91  suspicious         1.000         0.461   
9  Bruteforce-FTP       57.50  suspicious         0.998         0.217   

   z_component  
0        0.054  
1        0.049  
2        0.054  
3        0.054  
4        0.054  
5        1.000  
6   

In [19]:
# Cell 10 - SHAP Explainability with Plain-English Alerts
try:
    print("[Cell 10/13] Generating SHAP explanations...")

    explainer = shap.TreeExplainer(rf_model)

    feature_hint_map = {
        ("syn_flag_count", "high"): "Possible SYN flood behavior due to elevated SYN flags.",
        ("bwd_packets/s", "low"): "Possible one-way flood pattern with weak backward response.",
        ("flow_packets/s", "high"): "Very high packet rate suggests volumetric attack activity.",
        ("packet_length_variance", "high"): "High packet-size variance can indicate mixed malicious payload patterns.",
        ("idle_mean", "low"): "Low idle time suggests sustained aggressive traffic bursts.",
        ("active_mean", "high"): "High active window indicates persistent high activity flow behavior."
    }

    def to_shap_vector(shap_values_obj, sample_index: int, pred_class_index: int):
        # Handles both legacy list output and newer ndarray output across SHAP versions.
        if isinstance(shap_values_obj, list):
            return shap_values_obj[pred_class_index][sample_index]
        if isinstance(shap_values_obj, np.ndarray) and shap_values_obj.ndim == 3:
            return shap_values_obj[sample_index, :, pred_class_index]
        if isinstance(shap_values_obj, np.ndarray) and shap_values_obj.ndim == 2:
            return shap_values_obj[sample_index]
        raise ValueError("Unsupported SHAP output format.")

    x_explain = X_test_scaled[:5]
    preds_enc = rf_model.predict(x_explain)
    shap_values = explainer.shap_values(x_explain)

    example_explanations = []

    for i in range(x_explain.shape[0]):
        pred_cls = int(preds_enc[i])
        pred_label = label_encoder.inverse_transform([pred_cls])[0]

        sv = to_shap_vector(shap_values, i, pred_cls)
        top_idx = np.argsort(np.abs(sv))[-3:][::-1]

        top_triplets = []
        for j in top_idx:
            feat = selected_features[int(j)]
            direction = "high" if x_explain[i, int(j)] >= 0 else "low"
            desc = feature_hint_map.get((feat, direction), f"{direction.capitalize()} {feat} influenced this prediction.")
            top_triplets.append((feat, direction, float(sv[int(j)]), desc))

        plain_english = " ".join([t[3] for t in top_triplets])
        example_explanations.append({
            "prediction": pred_label,
            "top_3_features": [(t[0], t[1], round(t[2], 4)) for t in top_triplets],
            "plain_english_explanation": plain_english
        })

    print("5 SHAP explanation examples:")
    for idx, ex in enumerate(example_explanations, start=1):
        print(f"\nExample {idx}")
        print(f"Prediction: {ex['prediction']}")
        print(f"Top 3 features: {ex['top_3_features']}")
        print(f"Explanation: {ex['plain_english_explanation']}")
except Exception as e:
    print(f"[ERROR][Cell 10] {e}")
    raise

[Cell 10/13] Generating SHAP explanations...
5 SHAP explanation examples:

Example 1
Prediction: DoS-Hulk
Top 3 features: [('fwd_packet_length_max', 'high', 0.0977), ('packet_length_max', 'high', 0.0976), ('init_fwd_win_bytes', 'low', 0.0967)]
Explanation: High fwd_packet_length_max influenced this prediction. High packet_length_max influenced this prediction. Low init_fwd_win_bytes influenced this prediction.

Example 2
Prediction: DDoS-LDAP
Top 3 features: [('fwd_packet_length_mean', 'high', 0.1102), ('flow_bytes/s', 'low', 0.0961), ('packet_length_mean', 'high', 0.0832)]
Explanation: High fwd_packet_length_mean influenced this prediction. Low flow_bytes/s influenced this prediction. High packet_length_mean influenced this prediction.

Example 3
Prediction: DDoS-LOIC-HTTP
Top 3 features: [('init_fwd_win_bytes', 'low', 0.1151), ('subflow_fwd_bytes', 'low', 0.0869), ('avg_fwd_segment_size', 'low', 0.0707)]
Explanation: Low init_fwd_win_bytes influenced this prediction. Low subflow_fwd_

In [20]:
# Cell 11 - Convert Random Forest to ONNX + Validate
try:
    print("[Cell 11/13] Converting RF model to ONNX...")

    initial_type = [("float_input", FloatTensorType([None, len(selected_features)]))]

    # Disable ZipMap for leaner outputs and lower runtime overhead.
    rf_onnx = convert_sklearn(
        rf_model,
        initial_types=initial_type,
        target_opset=12,
        options={id(rf_model): {"zipmap": False}}
    )

    onnx_path = Path("rf_model.onnx")
    with open(onnx_path, "wb") as f:
        f.write(rf_onnx.SerializeToString())

    print(f"Saved {onnx_path}")

    session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    input_name = session.get_inputs()[0].name

    x_val = X_test_scaled[:200].astype(np.float32)

    sk_pred = rf_model.predict(x_val)

    t0 = time.perf_counter()
    onnx_outputs = session.run(None, {input_name: x_val})
    t1 = time.perf_counter()

    onnx_pred = np.asarray(onnx_outputs[0]).astype(sk_pred.dtype)
    match_ratio = float((onnx_pred == sk_pred).mean())
    infer_ms = (t1 - t0) * 1000.0

    print(f"ONNX vs sklearn prediction match ratio: {match_ratio:.4f}")
    print(f"ONNX inference time for 200 rows: {infer_ms:.3f} ms ({infer_ms / 200.0:.4f} ms/row)")
except Exception as e:
    print(f"[ERROR][Cell 11] {e}")
    raise

[Cell 11/13] Converting RF model to ONNX...
Saved rf_model.onnx
ONNX vs sklearn prediction match ratio: 1.0000
ONNX inference time for 200 rows: 2.645 ms (0.0132 ms/row)


In [21]:
# Cell 12 - Performance Benchmarking (Pi Constraints)
try:
    print("[Cell 12/13] Running performance benchmark...")

    sample_row = X_test_scaled[0:1].astype(np.float32)
    sample_flow_packets = np.array([flow_packets_s_test.iloc[0]], dtype=np.float32)

    # In-notebook timing (useful sanity check)
    t0 = time.perf_counter()
    _ = session.run(None, {input_name: sample_row})
    _ = iso_model.decision_function(sample_row)
    _ = compute_risk_scores(sample_row, sample_flow_packets)
    t1 = time.perf_counter()
    single_infer_ms = (t1 - t0) * 1000.0

    # Notebook RAM includes training data and is not deployment-representative.
    notebook_process = psutil.Process(os.getpid())
    notebook_ram_mb = notebook_process.memory_info().rss / (1024 ** 2)

    def file_size_mb(path_str: str):
        p = Path(path_str)
        return p.stat().st_size / (1024 ** 2) if p.exists() else np.nan

    rf_onnx_mb = file_size_mb("rf_model.onnx")
    iso_mb = file_size_mb("iso_model.pkl")
    scaler_mb = file_size_mb("scaler.pkl")
    le_mb = file_size_mb("label_encoder.pkl")

    # Measure deployment-like memory/time in a fresh subprocess that loads only artifacts.
    import json
    import subprocess
    import sys

    probe_script = r'''
import gc
import time
import json
import psutil
import numpy as np
import joblib
import onnxruntime as ort

scaler = joblib.load("scaler.pkl")
iso_model = joblib.load("iso_model.pkl")

# Aggressive low-memory ORT session options for edge deployment.
so = ort.SessionOptions()
so.enable_mem_pattern = False
so.enable_cpu_mem_arena = False
so.intra_op_num_threads = 1
so.inter_op_num_threads = 1
so.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
try:
    so.add_session_config_entry("session.disable_prepacking", "1")
except Exception:
    pass

session = ort.InferenceSession("rf_model.onnx", sess_options=so, providers=["CPUExecutionProvider"])
input_name = session.get_inputs()[0].name

x = np.zeros((1, scaler.n_features_in_), dtype=np.float32)
x_scaled = scaler.transform(x).astype(np.float32)

# Warmup + timed run
_ = session.run(None, {input_name: x_scaled})
_ = iso_model.decision_function(x_scaled)

t0 = time.perf_counter()
_ = session.run(None, {input_name: x_scaled})
_ = iso_model.decision_function(x_scaled)
t1 = time.perf_counter()

# Drop temporary arrays before memory probe.
del x, x_scaled, scaler
gc.collect()

time.sleep(0.15)
proc = psutil.Process()
mi = proc.memory_info()
try:
    uss_mb = proc.memory_full_info().uss / (1024 ** 2)
except Exception:
    uss_mb = None

print(json.dumps({
    "infer_ms": (t1 - t0) * 1000.0,
    "rss_mb": mi.rss / (1024 ** 2),
    "uss_mb": uss_mb
}))
'''

    cmd = [sys.executable, "-c", probe_script]
    probe = subprocess.run(cmd, capture_output=True, text=True, check=True)
    probe_out = json.loads(probe.stdout.strip())

    deploy_infer_ms = float(probe_out["infer_ms"])
    deploy_rss_mb = float(probe_out["rss_mb"])
    deploy_uss_mb = probe_out["uss_mb"]
    deploy_uss_mb = float(deploy_uss_mb) if deploy_uss_mb is not None else np.nan

    # Prefer USS (unique memory) when available; fallback to RSS.
    deploy_mem_metric = deploy_uss_mb if not np.isnan(deploy_uss_mb) else deploy_rss_mb

    print(f"Single-row inference time (notebook): {single_infer_ms:.3f} ms")
    print(f"Notebook RAM usage (includes training data): {notebook_ram_mb:.2f} MB")
    print(f"Single-row inference time (deployment-like subprocess): {deploy_infer_ms:.3f} ms")
    print(f"Deployment-like RSS RAM usage: {deploy_rss_mb:.2f} MB")
    if not np.isnan(deploy_uss_mb):
        print(f"Deployment-like USS RAM usage (unique memory): {deploy_uss_mb:.2f} MB")
    print("Model/artifact sizes (MB):")
    print(f"  rf_model.onnx: {rf_onnx_mb:.3f}")
    print(f"  iso_model.pkl: {iso_mb:.3f}")
    print(f"  scaler.pkl: {scaler_mb:.3f}")
    print(f"  label_encoder.pkl: {le_mb:.3f}")

    assert deploy_infer_ms < 50.0, f"Inference too slow: {deploy_infer_ms:.3f} ms >= 50 ms"
    assert deploy_mem_metric < 300.0, f"RAM too high: {deploy_mem_metric:.2f} MB >= 300 MB"
    assert rf_onnx_mb < 100.0, f"ONNX model too large: {rf_onnx_mb:.3f} MB >= 100 MB"

    print("All Raspberry Pi target constraints satisfied (deployment-like check).")
except subprocess.CalledProcessError as cpe:
    print("[ERROR][Cell 12] Failed to run deployment probe subprocess.")
    print(cpe.stderr)
    raise
except AssertionError as ae:
    print(f"[TARGET CHECK FAILED][Cell 12] {ae}")
    raise
except Exception as e:
    print(f"[ERROR][Cell 12] {e}")
    raise

[Cell 12/13] Running performance benchmark...
Single-row inference time (notebook): 39.353 ms
Notebook RAM usage (includes training data): 1656.29 MB
Single-row inference time (deployment-like subprocess): 4.442 ms
Deployment-like RSS RAM usage: 235.61 MB
Deployment-like USS RAM usage (unique memory): 179.43 MB
Model/artifact sizes (MB):
  rf_model.onnx: 14.375
  iso_model.pkl: 0.196
  scaler.pkl: 0.001
  label_encoder.pkl: 0.001
All Raspberry Pi target constraints satisfied (deployment-like check).


In [23]:
# Cell 13 - Build Prediction API and save predictor.py
try:
    print("[Cell 13/13] Creating predictor.py API...")

    predictor_code = r'''"""
Lightweight prediction API for CICIDS intrusion detection.
Returns attack type, risk score, confidence, SHAP top-3 features, and plain-English explanation.
"""

from pathlib import Path
import numpy as np
import joblib
import onnxruntime as ort
import shap

# Lazy-loaded global artifacts
_SCALER = None
_LABEL_ENCODER = None
_ISO_MODEL = None
_RF_MODEL = None
_ONNX_SESSION = None
_SELECTED_FEATURES = None
_INPUT_NAME = None
_EXPLAINER = None

# Cached stats for risk scoring
_ISO_MIN = None
_ISO_MAX = None
_FLOW_MU = 0.0
_FLOW_SIGMA = 1.0

_FEATURE_HINT_MAP = {
    ("syn_flag_count", "high"): "Possible SYN flood behavior due to elevated SYN flags.",
    ("bwd_packets/s", "low"): "Possible one-way flood pattern with weak backward response.",
    ("flow_packets/s", "high"): "Very high packet rate suggests volumetric attack activity.",
    ("packet_length_variance", "high"): "High packet-size variance can indicate mixed malicious payload patterns.",
    ("idle_mean", "low"): "Low idle time suggests sustained aggressive traffic bursts.",
    ("active_mean", "high"): "High active window indicates persistent high activity flow behavior."
}


def _load_selected_features(path="selected_features.txt"):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Missing feature list: {p.resolve()}")
    return [line.strip() for line in p.read_text(encoding="utf-8").splitlines() if line.strip()]


def _ensure_loaded():
    global _SCALER, _LABEL_ENCODER, _ISO_MODEL, _RF_MODEL, _ONNX_SESSION
    global _SELECTED_FEATURES, _INPUT_NAME, _EXPLAINER, _ISO_MIN, _ISO_MAX

    if _SCALER is None:
        _SCALER = joblib.load("scaler.pkl")
    if _LABEL_ENCODER is None:
        _LABEL_ENCODER = joblib.load("label_encoder.pkl")
    if _ISO_MODEL is None:
        _ISO_MODEL = joblib.load("iso_model.pkl")
    if _RF_MODEL is None:
        _RF_MODEL = joblib.load("rf_model.pkl")
    if _SELECTED_FEATURES is None:
        _SELECTED_FEATURES = _load_selected_features("selected_features.txt")
    if _ONNX_SESSION is None:
        _ONNX_SESSION = ort.InferenceSession("rf_model.onnx", providers=["CPUExecutionProvider"])
        _INPUT_NAME = _ONNX_SESSION.get_inputs()[0].name
    if _EXPLAINER is None:
        _EXPLAINER = shap.TreeExplainer(_RF_MODEL)

    # Build normalization range once using scaler statistics around zero-centered synthetic point.
    if _ISO_MIN is None or _ISO_MAX is None:
        ref = np.zeros((256, len(_SELECTED_FEATURES)), dtype=np.float32)
        ref += np.random.normal(0.0, 1.0, size=ref.shape).astype(np.float32)
        iso_vals = -_ISO_MODEL.decision_function(ref)
        _ISO_MIN = float(np.min(iso_vals))
        _ISO_MAX = float(np.max(iso_vals))


def _to_shap_vector(shap_values_obj, pred_class_index):
    if isinstance(shap_values_obj, list):
        return np.asarray(shap_values_obj[pred_class_index][0])
    if isinstance(shap_values_obj, np.ndarray) and shap_values_obj.ndim == 3:
        return np.asarray(shap_values_obj[0, :, pred_class_index])
    if isinstance(shap_values_obj, np.ndarray) and shap_values_obj.ndim == 2:
        return np.asarray(shap_values_obj[0])
    raise ValueError("Unsupported SHAP output format.")


def _explain(top_features):
    fragments = []
    for feat, direction, _ in top_features:
        fragments.append(_FEATURE_HINT_MAP.get((feat, direction), f"{direction.capitalize()} {feat} influenced this decision."))
    return " ".join(fragments)


def predict(flow_features_dict):
    """
    Parameters
    ----------
    flow_features_dict : dict
        Dictionary containing CICIDS feature names and numeric values.

    Returns
    -------
    dict
        {
          "attack_type": str,
          "risk_score": float,
          "confidence": float,
          "top_3_shap_features": list,
          "plain_english_explanation": str
        }
    """
    _ensure_loaded()

    row = []
    for feat in _SELECTED_FEATURES:
        val = flow_features_dict.get(feat, 0.0)
        try:
            row.append(float(val))
        except Exception:
            row.append(0.0)

    x = np.array([row], dtype=np.float32)
    x_scaled = _SCALER.transform(x).astype(np.float32)

    onnx_out = _ONNX_SESSION.run(None, {_INPUT_NAME: x_scaled})
    pred_enc = int(np.asarray(onnx_out[0])[0])
    pred_label = _LABEL_ENCODER.inverse_transform([pred_enc])[0]

    # Probability output can be a list of dicts depending on ONNX conversion options.
    proba_raw = onnx_out[1]
    if isinstance(proba_raw, list) and len(proba_raw) > 0 and isinstance(proba_raw[0], dict):
        proba_vec = np.zeros(len(_LABEL_ENCODER.classes_), dtype=np.float32)
        for k, v in proba_raw[0].items():
            proba_vec[int(k)] = float(v)
    else:
        proba_vec = np.asarray(proba_raw)[0].astype(np.float32)

    benign_candidates = np.where(_LABEL_ENCODER.classes_ == "Benign")[0]
    benign_idx = int(benign_candidates[0]) if len(benign_candidates) else 0

    rf_attack_prob = 1.0 - float(proba_vec[benign_idx])

    iso_raw = float(-_ISO_MODEL.decision_function(x_scaled)[0])
    iso_norm = (iso_raw - _ISO_MIN) / (_ISO_MAX - _ISO_MIN + 1e-8)
    iso_norm = float(np.clip(iso_norm, 0.0, 1.0))

    flow_packets_val = float(flow_features_dict.get("flow_packets/s", 0.0))
    z = abs((flow_packets_val - _FLOW_MU) / (_FLOW_SIGMA + 1e-8))
    z_norm = float(np.clip(z / 3.0, 0.0, 1.0))

    risk_score = float(np.clip((0.5 * rf_attack_prob + 0.3 * iso_norm + 0.2 * z_norm) * 100.0, 0.0, 100.0))

    shap_values = _EXPLAINER.shap_values(x_scaled)
    sv = _to_shap_vector(shap_values, pred_enc)
    top_idx = np.argsort(np.abs(sv))[-3:][::-1]

    top_3 = []
    for i in top_idx:
        feat = _SELECTED_FEATURES[int(i)]
        direction = "high" if float(x_scaled[0, int(i)]) >= 0 else "low"
        top_3.append((feat, direction, float(sv[int(i)])))

    return {
        "attack_type": str(pred_label),
        "risk_score": round(risk_score, 2),
        "confidence": round(float(np.max(proba_vec)), 4),
        "top_3_shap_features": [(f, d, round(v, 4)) for f, d, v in top_3],
        "plain_english_explanation": _explain(top_3)
    }
'''

    with open("predictor.py", "w", encoding="utf-8") as f:
        f.write(predictor_code)

    print("Saved predictor.py")

    # Quick smoke test with one test row converted back to dict format.
    sample_dict = {feat: float(X_test.iloc[0][feat]) for feat in selected_features}

    # Persist flow stat values into module file for z-score realism.
    flow_mu_local = float(flow_packets_s_train.mean())
    flow_sigma_local = float(flow_packets_s_train.std()) + 1e-8

    with open("predictor.py", "a", encoding="utf-8") as f:
        f.write(f"\n_FLOW_MU = {flow_mu_local}\n")
        f.write(f"_FLOW_SIGMA = {flow_sigma_local}\n")

    import importlib.util
    spec = importlib.util.spec_from_file_location("predictor", "predictor.py")
    predictor = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(predictor)

    pred_out = predictor.predict(sample_dict)
    print("Sample predictor output:")
    print(pred_out)
except Exception as e:
    print(f"[ERROR][Cell 13] {e}")
    raise

[Cell 13/13] Creating predictor.py API...
Saved predictor.py
Sample predictor output:
{'attack_type': 'DoS-Hulk', 'risk_score': 53.23, 'confidence': 0.96, 'top_3_shap_features': [('fwd_packet_length_max', 'high', 0.0977), ('packet_length_max', 'high', 0.0976), ('init_fwd_win_bytes', 'low', 0.0967)], 'plain_english_explanation': 'High fwd_packet_length_max influenced this decision. High packet_length_max influenced this decision. Low init_fwd_win_bytes influenced this decision.'}
